In [1]:


from utils.huggingface import suggest_automodel_class
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from exp.baselines import DNNmem

/home/mechrevo/miniconda3/envs/xMem-LLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Transformers Model

In [2]:
model_name = "facebook/opt-125m"
model_class = suggest_automodel_class(model_name)
model = model_class.from_pretrained(model_name)

/home/mechrevo/miniconda3/envs/xMem-LLM/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/mechrevo/miniconda3/envs/xMem-LLM/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# 2. 加载数据集
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. 分词
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 4. 创建 DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. 创建 DataLoader
tokenized_datasets.set_format("torch")
dataloader = DataLoader(tokenized_datasets, batch_size=10, shuffle=True, collate_fn=data_collator)


Map: 100%|██████████| 36718/36718 [00:02<00:00, 13208.46 examples/s]


# CNN Model

In [4]:
from perf_estimator.models import AllModels
from perf_estimator.dataset import image_dataset

In [5]:
cnn_model = AllModels["ResNet50"].value
cnn_dl = image_dataset()

Files already downloaded and verified


# Estimation

## DNNmem

In [6]:
cnn_est = DNNmem(
    model=cnn_model,
    dataloader=cnn_dl,
    max_est_memory_in_bytes=8*1024*1024*1024,  # 8GB
    optimizer=torch.optim.SGD,
    is_transformer=False
)
cnn_est.estimate()
print(f"Estimated memory: {cnn_est.estimate_memory} bytes, and execute time: {cnn_est.execute_time} ns")


Execute the forward and backward analysis first.
Estimated memory: 3189768192 bytes, and execute time: 28653738265 ns


In [7]:
transformer = DNNmem(
    model=model,
    dataloader=dataloader,
    max_est_memory_in_bytes=8*1024*1024*1024,  # 8GB
    optimizer=torch.optim.AdamW,
    is_transformer=True
)
transformer.estimate()
print(f"Estimated memory: {transformer.estimate_memory} bytes, and execute time: {transformer.execute_time} ns")

Estimated memory: 3206545408 bytes, and execute time: 1157493736 ns


## Schedtune

In [6]:
from exp.baselines.schedtune import ScheduleTune
cnn_sched = ScheduleTune(
    model=cnn_model,
    dataloader=cnn_dl,
    optimizer=torch.optim.SGD,
    is_transformer=False,
    device_id=0,
)
cnn_sched.estimate()
print(f"Estimated memory: {cnn_sched.estimate_memory} bytes, and execute time: {cnn_sched.execute_time} ns")

4742.340000,147.144673
Estimated memory: 4972703907.84 bytes, and execute time: 42723883 ns


In [7]:
llm_sched = ScheduleTune(
    model=model,
    dataloader=dataloader,
    optimizer=torch.optim.AdamW,
    is_transformer=True,
    device_id=0,
)
llm_sched.estimate()
print(f"Estimated memory: {llm_sched.estimate_memory} bytes, and execute time: {llm_sched.execute_time} ns")

3955.330000,134.915229
Estimated memory: 4147464110.08 bytes, and execute time: 20745637 ns


In [1]:
from exp.run import ExperimentRun
ExperimentRun().run_cnn_experiment()

Files already downloaded and verified
Starting estimation for VGG11-SGD-50
Execute the forward and backward analysis first.
Model: VGG11-SGD50, Memory: 1503657984, Runtime: 23915497569
Files already downloaded and verified
Starting estimation for VGG11-SGD-51
Execute the forward and backward analysis first.
Model: VGG11-SGD51, Memory: 1512046592, Runtime: 24253117862
Files already downloaded and verified
Starting estimation for VGG11-SGD-52
Execute the forward and backward analysis first.
Model: VGG11-SGD52, Memory: 1518338048, Runtime: 24576431233
Files already downloaded and verified
Starting estimation for VGG11-SGD-53
Execute the forward and backward analysis first.
Model: VGG11-SGD53, Memory: 1526726656, Runtime: 24361690402
Files already downloaded and verified
Starting estimation for VGG11-SGD-54
Execute the forward and backward analysis first.
Model: VGG11-SGD54, Memory: 1537212416, Runtime: 24302234293
Files already downloaded and verified
Starting estimation for VGG11-SGD-55


KeyboardInterrupt: 